# FinanceAI — Dataset de usuarios

Este notebook genera el archivo base `users.csv` para la simulación de transacciones.

### Definiciones importantes

- `meta_ahorro`: porcentaje de ingreso que el usuario **declara** que desea ahorrar. No representa el ahorro real.
- `frecuencia_ahorro`: frecuencia declarada por el usuario.
- `ratio_deuda_inicial`: deuda inicial expresada como proporción del ingreso mensual estimado. Por ejemplo, `0.35` equivale al 35 %.
- `saldo_deuda_inicial`: monto inicial de deuda calculado a partir del ingreso estimado y el ratio anterior.
- `arquetipo_comportamiento`: variable latente utilizada para generar datos sintéticos. Debe excluirse de las variables de entrada de los modelos finales para evitar fuga de información.


In [54]:
# =========================
# LIBRERÍAS Y CONFIGURACIÓN
# =========================

from pathlib import Path
import random

import numpy as np
import pandas as pd

SEED = 42
NUM_USERS = 300
OUTPUT_FILENAME = "users.csv"

# Reproducibilidad
random.seed(SEED)
np.random.seed(SEED)

# Funciona tanto en Google Colab como en ejecución local.
OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path(".")
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FILENAME

print(f"Semilla: {SEED}")
print(f"Usuarios a generar: {NUM_USERS}")
print(f"Archivo de salida: {OUTPUT_PATH.resolve()}")


Semilla: 42
Usuarios a generar: 300
Archivo de salida: /content/users.csv


In [55]:
# ========
# CIUDADES
# ========

CITIES = ["Aguascalientes", "Mexicali", "La Paz", "Campeche", "Tuxtla Gutiérrez",
          "Chihuahua", "Saltillo", "Colima", "Durango", "Guanajuato", "Chilpancingo",
          "Pachuca", "Guadalajara", "Toluca", "Morelia", "Cuernavaca", "Tepic",
          "Monterrey", "Oaxaca", "Puebla", "Querétaro", "Chetumal", "San Luis Potosí",
          "Culiacán", "Hermosillo", "Villahermosa", "Ciudad Victoria", "Tlaxcala",
          "Xalapa", "Mérida", "Zacatecas"]



In [56]:
# ================================
# OCUPACIONES CON REGLAS REALISTAS
# ================================


OCCUPATIONS = {
    "Estudiante": {"edad": (18, 25), "ingreso": (3000, 10000)},
    "Arquitecto": {"edad": (25, 60), "ingreso": (15000, 30000)},
    "Ingeniero": {"edad": (23, 60), "ingreso": (15000, 50000)},
    "Médico": {"edad": (28, 70), "ingreso": (15000, 60000)},
    "Dentista": {"edad": (25, 65), "ingreso": (15000, 30000)},
    "Abogado": {"edad": (25, 70), "ingreso": (15000, 30000)},
    "Profesor": {"edad": (23, 70), "ingreso": (12000, 35000)},
    "Contador": {"edad": (23, 65), "ingreso": (12000, 30000)},
    "Programador": {"edad": (20, 55), "ingreso": (20000, 60000)},
    "Analista de Datos": {"edad": (22, 55), "ingreso": (18000, 50000)},
    "Diseñador": {"edad": (20, 55), "ingreso": (12000, 35000)},
    "Enfermero": {"edad": (22, 65), "ingreso": (15000, 40000)},
    "Chef": {"edad": (20, 60), "ingreso": (12000, 60000)},
    "Veterinario": {"edad": (25, 65), "ingreso": (12000, 70000)},
    "Comerciante": {"edad": (25, 70), "ingreso": (12000, 80000)},
    "Emprendedor": {"edad": (25, 65), "ingreso": (15000, 120000)},
    "Freelancer": {"edad": (20, 60), "ingreso": (10000, 70000)},
    "Ejecutivo": {"edad": (30, 65), "ingreso": (50000, 150000)},
    "Jubilado": {"edad": (60, 80), "ingreso": (10000, 40000)},
}

VARIABLE_INCOME_MAX = {
    "Estudiante": 0.10,
    "Arquitecto": 0.20,
    "Ingeniero": 0.10,
    "Médico": 0.20,
    "Dentista": 0.20,
    "Abogado": 0.20,
    "Profesor": 0.10,
    "Contador": 0.10,
    "Programador": 0.20,
    "Analista de Datos": 0.10,
    "Diseñador": 0.30,
    "Enfermero": 0.10,
    "Chef": 0.30,
    "Veterinario": 0.20,
    "Comerciante": 0.50,
    "Emprendedor": 0.70,
    "Freelancer": 0.60,
    "Ejecutivo": 0.20,
    "Jubilado": 0.05,
}


In [57]:
# ======================================================
# ARQUETIPOS DE COMPORTAMIENTO FINANCIERO
# ======================================================

ARCHETYPE_BEHAVIOR = {

    "Ahorrador": {
        "peso": 0.20,
        "ahorro_real": (0.15, 0.30),
        "deuda": (0.05, 0.20),
        "control_gasto": (0.75, 0.95),
        "gasto_discrecional": (0.05, 0.15)
    },

    "Equilibrado": {
        "peso": 0.30,
        "ahorro_real": (0.08, 0.18),
        "deuda": (0.10, 0.35),
        "control_gasto": (0.60, 0.85),
        "gasto_discrecional": (0.10, 0.25)
    },

    "Gastador": {
        "peso": 0.20,
        "ahorro_real": (0.01, 0.08),
        "deuda": (0.15, 0.50),
        "control_gasto": (0.25, 0.55),
        "gasto_discrecional": (0.25, 0.45)
    },

    "Endeudado": {
        "peso": 0.15,
        "ahorro_real": (0.00, 0.07),
        "deuda": (0.40, 0.75),
        "control_gasto": (0.35, 0.65),
        "gasto_discrecional": (0.05, 0.20)
    },

    "Financieramente_Inestable": {
        "peso": 0.15,
        "ahorro_real": (0.00, 0.20),
        "deuda": (0.10, 0.70),
        "control_gasto": (0.20, 0.60),
        "gasto_discrecional": (0.10, 0.50)
    }
}

ARCHETYPE_NAMES = list(ARCHETYPE_BEHAVIOR)
ARCHETYPE_WEIGHTS = [ARCHETYPE_BEHAVIOR[name]["peso"] for name in ARCHETYPE_NAMES]

assert np.isclose(sum(ARCHETYPE_WEIGHTS), 1.0), "Los pesos de arquetipos deben sumar 1."


In [58]:
# ====================
# FUNCIONES AUXILIARES
# ====================

# SITUACIÓN DE VIDA

def generate_life_situation(edad: int) -> str:
    if edad <= 22:
        opciones = ["Vive con familia", "Vive solo", "Comparte vivienda"]
        pesos = [0.60, 0.20, 0.20]
    elif edad <= 30:
        opciones = ["Vive solo", "Vive con pareja", "Vive con familia", "Comparte vivienda"]
        pesos = [0.30, 0.25, 0.20, 0.25]
    elif edad <= 50:
        opciones = ["Vive solo", "Vive con pareja", "Vive con familia"]
        pesos = [0.20, 0.40, 0.40]
    else:
        opciones = ["Vive solo", "Vive con pareja", "Vive con familia"]
        pesos = [0.30, 0.50, 0.20]

    return random.choices(opciones, weights=pesos, k=1)[0]


# SELECCIÓN DE ARQUETIPO

def select_archetype() -> str: return random.choices(population=ARCHETYPE_NAMES,weights=ARCHETYPE_WEIGHTS,k=1,)[0]

## Generación de usuarios sintéticos

El dataset relaciona ocupación, edad, ingreso y situación de vida mediante reglas de coherencia.

- `ratio_deuda_inicial` se guarda como decimal entre `0.05` y `0.60`.
- `saldo_deuda_inicial` se guarda como monto en pesos mexicanos y equivale a `ingreso_total_estimado × ratio_deuda_inicial`.

El objetivo es generar datos sintéticos realistas que permitan posteriormente simular transacciones financieras y entrenar modelos de clasificación y análisis de salud financiera.

In [59]:
# ==============================
# GENERADOR DE USUARIO COHERENTE
# ==============================

def generate_user(user_number: int) -> dict:
    # Ocupación, edad e ingreso base
    ocupacion = random.choice(list(OCCUPATIONS))
    reglas = OCCUPATIONS[ocupacion]
    edad = random.randint(*reglas["edad"])
    ingreso_base = round(random.uniform(*reglas["ingreso"]), 2)

    # Ingreso variable según la estabilidad habitual de la ocupación
    porcentaje_variable = random.uniform(0, VARIABLE_INCOME_MAX[ocupacion])
    ingreso_variable = round(ingreso_base * porcentaje_variable, 2)
    ingreso_total_estimado = round(ingreso_base + ingreso_variable, 2)

    # Información personal y declarada
    situacion_vida = generate_life_situation(edad)
    meta_ahorro = round(random.uniform(5, 30), 2)  # porcentaje deseado establecido
    frecuencia_ahorro = random.choice(["Alta", "Media", "Baja"])

    # Deuda inicial con unidad explícita
    ratio_deuda_inicial = round(random.uniform(0.05, 0.60), 4)
    saldo_deuda_inicial = round(ingreso_total_estimado * ratio_deuda_inicial, 2)

    # Variable latente para generar el historial sintético
    # (El arquetipo NO determina la meta de ahorro declarada por el usuario. Su función es representar un patrón de comportamiento que utilizaremos posteriormente para generar las transacciones sintéticas.)

    arquetipo = select_archetype()

    return {
        "user_id": f"USR_{user_number:05d}",
        "edad": edad,
        "sexo": random.choice(["M", "F", "O"]),
        "ocupacion": ocupacion,
        "situacion_vida": situacion_vida,
        "ciudad": random.choice(CITIES),
        "ingreso_base": ingreso_base,
        "ingreso_variable": ingreso_variable,
        "ingreso_total_estimado": ingreso_total_estimado,
        "meta_ahorro": meta_ahorro,
        "frecuencia_ahorro": frecuencia_ahorro,
        "ratio_deuda_inicial": ratio_deuda_inicial,
        "saldo_deuda_inicial": saldo_deuda_inicial,
        "arquetipo_comportamiento": arquetipo,
    }


In [60]:
# ===========================
# GENERAR DATASET DE USUARIOS
# ===========================

users = [generate_user(i)for i in range(1,NUM_USERS + 1)]
users_df = pd.DataFrame(users)
users_df.sample(5)

,user_id,edad,sexo,ocupacion,situacion_vida,ciudad,ingreso_base,ingreso_variable,ingreso_total_estimado,meta_ahorro,frecuencia_ahorro,ratio_deuda_inicial,saldo_deuda_inicial,arquetipo_comportamiento
203,USR_00204,48,F,Analista de Datos,Vive con familia,Tlaxcala,23759.40,321.77,24081.17,23.23,Media,0.2560,6164.78,Financieramente_Inestable
266,USR_00267,26,M,Comerciante,Vive con pareja,Xalapa,20448.13,8768.69,29216.82,5.02,Media,0.1613,4712.67,Gastador
152,USR_00153,22,F,Programador,Vive con familia,Toluca,50131.64,1916.07,52047.71,24.52,Baja,0.2326,12106.30,Ahorrador
9,USR_00010,31,O,Contador,Vive solo,La Paz,21171.47,192.47,21363.94,7.74,Baja,0.1380,2948.22,Gastador
233,USR_00234,65,F,Ejecutivo,Vive solo,Aguascalientes,101244.08,2091.56,103335.64,18.86,Baja,0.3789,39153.87,Equilibrado


In [61]:
# ==============================
# VALIDACIÓN GENERAL DEL DATASET
# ==============================

print("Número de usuarios:",len(users_df))
print("\nColumnas:")
print(users_df.columns.tolist())
print("\nValores nulos:")
print(users_df.isnull().sum())
print("\nDistribución de arquetipos:")
print(users_df["arquetipo_comportamiento"].value_counts())
print("\nDistribución porcentual:")
print((users_df["arquetipo_comportamiento"].value_counts(normalize=True)* 100).round(2))

Número de usuarios: 300

Columnas:
['user_id', 'edad', 'sexo', 'ocupacion', 'situacion_vida', 'ciudad', 'ingreso_base', 'ingreso_variable', 'ingreso_total_estimado', 'meta_ahorro', 'frecuencia_ahorro', 'ratio_deuda_inicial', 'saldo_deuda_inicial', 'arquetipo_comportamiento']

Valores nulos:
user_id                     0
edad                        0
sexo                        0
ocupacion                   0
situacion_vida              0
ciudad                      0
ingreso_base                0
ingreso_variable            0
ingreso_total_estimado      0
meta_ahorro                 0
frecuencia_ahorro           0
ratio_deuda_inicial         0
saldo_deuda_inicial         0
arquetipo_comportamiento    0
dtype: int64

Distribución de arquetipos:
arquetipo_comportamiento
Equilibrado                  80
Ahorrador                    72
Gastador                     54
Endeudado                    50
Financieramente_Inestable    44
Name: count, dtype: int64

Distribución porcentual:
arquetipo

In [65]:
# ===================
# RESUMEN DEL DATASET
# ===================

print("\nResumen de variables financieras:")
display(
    users_df[
        [
            "ingreso_base",
            "ingreso_variable",
            "ingreso_total_estimado",
            "meta_ahorro",
            "ratio_deuda_inicial",
            "saldo_deuda_inicial",
        ]
    ].describe().round(2)
)


Resumen de variables financieras:


,ingreso_base,ingreso_variable,ingreso_total_estimado,meta_ahorro,ratio_deuda_inicial,saldo_deuda_inicial
count,300.00,300.00,300.00,300.00,300.00,300.00
mean,36191.94,4487.35,40679.29,16.97,0.33,14121.59
std,26253.05,6276.60,30199.95,6.75,0.16,14683.51
min,3167.37,2.70,3338.90,5.02,0.06,497.27
25%,19653.44,893.10,21596.34,11.60,0.20,5084.31
50%,26919.15,2198.13,30037.06,16.56,0.34,10394.77
75%,45398.53,4689.17,51167.24,22.31,0.47,16536.48
max,147383.70,43372.22,169475.86,29.84,0.60,99465.38


In [67]:
# ================
# EXPORTAR DATASET
# ================

users_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("users.csv generado correctamente.")
print(f"Ruta: {OUTPUT_PATH.resolve()}")
print(f"Dimensiones: {users_df.shape}")


users.csv generado correctamente.
Ruta: /content/users.csv
Dimensiones: (300, 14)
